# Phase 1 Final Calibration

This notebook is an orchestration and audit surface only. Durable scientific logic lives in `src/` and is exercised through `python -m src.cli phase1_final_calibration`.

Scientific integrity boundary:

- This step consumes a reviewed final comparator reconciliation run and final comparator logits.
- It computes calibration diagnostics only from existing final logits: pooled ECE, subject ECE, Brier score, negative log-likelihood, reliability tables, reliability diagram data, risk-coverage curves and delta ECE against the locked baseline.
- It does not recalibrate predictions, retrain models, alter logits, fabricate diagrams, promote smoke outputs or open claims.
- If calibration threshold checks fail, the correct result is blocked/non-claim.
- This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy or full Phase 1 performance.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from __future__ import annotations

import getpass
import hashlib
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
RUN_UNITTESTS = True


def run(cmd, cwd=None, check=True):
    printable = ['<redacted>' if 'Authorization:' in str(part) else str(part) for part in cmd]
    print('$', ' '.join(printable))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)


if not REPO_DIR.exists():
    token = getpass.getpass('Enter GitHub token for private repo, or leave blank for public clone: ')
    if token:
        header = 'Authorization: Basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

if RUN_UNITTESTS:
    unit = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, text=True)
    if unit.returncode != 0:
        raise subprocess.CalledProcessError(unit.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
COMPARATOR_RECONCILIATION_RUN = None  # Optional explicit Path(...); defaults to latest artifact.
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_calibration'
PLAN_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_calibration_plan'

RUN_FINAL_CALIBRATION = False
REQUIRED_ACK = 'I reviewed final calibration and understand it computes diagnostics only from final logits without opening claims'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_calibration_v1_20260422'


def latest_run(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        candidate = Path(latest.read_text(encoding='utf-8').strip())
        if candidate.exists():
            return candidate
    runs = sorted([path for path in root.iterdir() if path.is_dir()])
    if not runs:
        raise FileNotFoundError(f'No runs under {root}')
    return runs[-1]


if COMPARATOR_RECONCILIATION_RUN is None:
    COMPARATOR_RECONCILIATION_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_comparator_reconciliation')

print('Notebook fix marker:', FIX_MARKER)
print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)
print('RUN_FINAL_CALIBRATION:', RUN_FINAL_CALIBRATION)


In [ ]:
# Cell 3 - Resolve paths and validate prereg identity plus comparator reconciliation boundary.


def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


assert PREREG_BUNDLE.exists(), f'Missing prereg bundle: {PREREG_BUNDLE}'
assert COMPARATOR_RECONCILIATION_RUN.exists(), f'Missing comparator reconciliation run: {COMPARATOR_RECONCILIATION_RUN}'

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

reconciliation_summary = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciliation_summary.json')
reconciliation_runtime = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciled_runtime_leakage_audit.json')
reconciliation_claim_state = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciled_claim_state.json')

assert reconciliation_summary.get('status') == 'phase1_final_comparator_reconciliation_complete_claim_closed'
assert reconciliation_summary.get('all_final_comparator_outputs_present') is True
assert reconciliation_runtime.get('runtime_logs_audited_for_all_required_comparators') is True
assert reconciliation_summary.get('claim_ready') is False
assert reconciliation_claim_state.get('claim_ready') is False

print('Completed comparators:', reconciliation_summary.get('completed_comparators'))
print('Runtime logs audited:', reconciliation_runtime.get('runtime_logs_audited_for_all_required_comparators'))
print('Claim ready:', reconciliation_summary.get('claim_ready'))


In [ ]:
# Cell 4 - Preflight runner/config availability and write a reviewable plan artifact.

created_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir = PLAN_ROOT / created_utc
plan_dir.mkdir(parents=True, exist_ok=False)

required_repo_paths = [
    REPO_DIR / 'src/phase1/final_calibration.py',
    REPO_DIR / 'src/phase1/calibration.py',
    REPO_DIR / 'configs/phase1/final_calibration.json',
    REPO_DIR / 'configs/eval/metrics.yaml',
    REPO_DIR / 'configs/eval/inference_defaults.yaml',
    REPO_DIR / 'configs/gate1/decision_simulation.json',
]
preflight_blockers = [str(path.relative_to(REPO_DIR)) + ' missing' for path in required_repo_paths if not path.exists()]
help_result = subprocess.run(['python', '-m', 'src.cli', 'phase1_final_calibration', '--help'], cwd=REPO_DIR, text=True, capture_output=True)
runner_available = help_result.returncode == 0 and 'phase1_final_calibration' in help_result.stdout
if not runner_available:
    preflight_blockers.append('CLI phase1_final_calibration route unavailable')

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_calibration',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--comparator-reconciliation-run', str(COMPARATOR_RECONCILIATION_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--calibration-config', 'configs/phase1/final_calibration.json',
    '--metrics-config', 'configs/eval/metrics.yaml',
    '--inference-config', 'configs/eval/inference_defaults.yaml',
    '--gate1-config', 'configs/gate1/decision_simulation.json',
]

plan = {
    'status': 'phase1_final_calibration_plan_recorded',
    'created_utc': created_utc,
    'notebook_fix_marker': FIX_MARKER,
    'run_flag': RUN_FINAL_CALIBRATION,
    'runner_available': runner_available,
    'preflight_blockers': preflight_blockers,
    'expected_locked_prereg_identity_hash': EXPECTED_PREREG_IDENTITY_HASH,
    'locked_prereg_hash_from_bundle': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_file_sha256,
    'comparator_reconciliation_run': str(COMPARATOR_RECONCILIATION_RUN),
    'command': cmd,
    'scientific_integrity_boundary': [
        'Calibration diagnostics are computed only from final comparator logits.',
        'The runner must not recalibrate, retrain, edit logits, fabricate diagrams or open claims.',
        'If calibration thresholds fail, the correct result is blocked/non-claim.',
    ],
}
(plan_dir / 'phase1_final_calibration_plan.json').write_text(json.dumps(plan, indent=2) + '
', encoding='utf-8')
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual hold or launch the CLI-backed final calibration runner.

if not RUN_FINAL_CALIBRATION:
    hold = {
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'status': 'phase1_final_calibration_manual_hold',
        'run_flag': RUN_FINAL_CALIBRATION,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'plan_dir': str(plan_dir),
        'preflight_blockers': preflight_blockers,
        'scientific_limit': 'Manual hold only; no calibration diagnostics were computed.',
    }
    (plan_dir / 'phase1_final_calibration_manual_hold.json').write_text(json.dumps(hold, indent=2) + '
', encoding='utf-8')
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch final calibration without explicit non-claim acknowledgement.')
elif preflight_blockers:
    raise RuntimeError(f'Preflight blockers remain: {preflight_blockers}')
else:
    run(cmd, cwd=REPO_DIR)
    print('Final calibration command completed. Review generated artifacts before downstream governance reconciliation.')


In [ ]:
# Cell 6 - Inspect latest final calibration output if execution was launched.

latest_run = None
summary = None
manifest = None
if RUN_FINAL_CALIBRATION:
    latest_file = OUTPUT_ROOT / 'latest.txt'
    assert latest_file.exists(), f'Missing latest pointer: {latest_file}'
    latest_run = Path(latest_file.read_text(encoding='utf-8').strip())
    print('Latest final calibration output:', latest_run)
    required_files = [
        'phase1_final_calibration_inputs.json',
        'phase1_final_calibration_summary.json',
        'phase1_final_calibration_report.md',
        'phase1_final_calibration_source_links.json',
        'phase1_final_calibration_input_validation.json',
        'final_comparator_logits_index.json',
        'pooled_ece_10_bins.json',
        'subject_level_ece.json',
        'brier_score.json',
        'negative_log_likelihood.json',
        'reliability_table.json',
        'reliability_diagram.json',
        'risk_coverage_curve.json',
        'calibration_delta_vs_baseline.json',
        'final_calibration_manifest.json',
        'phase1_final_calibration_claim_state.json',
    ]
    for name in required_files:
        path = latest_run / name
        print(name, 'exists =', path.exists())
        assert path.exists(), f'Missing expected artifact: {path}'
    summary = read_json(latest_run / 'phase1_final_calibration_summary.json')
    manifest = read_json(latest_run / 'final_calibration_manifest.json')
    print(json.dumps(summary, indent=2))
else:
    print('No execution launched. Plan source of truth:', plan_dir)


In [ ]:
# Cell 7 - Assertions and non-claim review note.

if RUN_FINAL_CALIBRATION:
    assert summary is not None and manifest is not None
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert manifest.get('claim_ready') is False
    assert manifest.get('smoke_artifacts_promoted') is False
    assert set(manifest.get('artifacts', [])) == set(manifest.get('required_artifacts', []))
    if manifest.get('calibration_package_passed') is not True:
        assert summary.get('status') == 'phase1_final_calibration_blocked'
        assert summary.get('claim_blockers'), 'Blocked calibration must name blockers.'

    review_note = {
        'status': 'phase1_final_calibration_review_recorded_claim_closed',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(latest_run),
        'calibration_package_passed': manifest.get('calibration_package_passed'),
        'checks_recorded': [
            'required_artifacts_present',
            'claim_ready_false',
            'smoke_artifacts_not_promoted',
            'diagnostics_computed_from_final_logits_only',
            'threshold_result_recorded_without_opening_claims',
        ],
        'allowed_interpretation': 'Final calibration diagnostics and threshold status only.',
        'not_allowed_interpretation': 'Do not treat calibration pass/fail as decoder efficacy or privileged-transfer evidence.',
        'not_ok_to_claim': [
            'decoder efficacy',
            'A2d efficacy',
            'A3/A4 efficacy',
            'A4 superiority',
            'privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    }
    note_path = latest_run / 'phase1_final_calibration_review_note.json'
    note_path.write_text(json.dumps(review_note, indent=2) + '
', encoding='utf-8')
    print('Review note written:', note_path)
    print(json.dumps(review_note, indent=2))
else:
    assert RUN_FINAL_CALIBRATION is False
    print('Manual hold verified. No final calibration artifacts were generated by this pass.')


In [ ]:
# Cell 8 - Closeout.

print('================ Phase 1 Final Calibration Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_CALIBRATION)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)
print('Preflight blockers:', preflight_blockers)
if RUN_FINAL_CALIBRATION:
    print('Latest final calibration output:', latest_run)
    print('Calibration package passed:', summary.get('calibration_package_passed'))
    print('Max allowed delta ECE:', summary.get('max_allowed_delta_ece'))
    print('Max abs delta ECE vs baseline:', summary.get('max_abs_delta_ece_vs_baseline'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review final_calibration_manifest.json and calibration threshold status before rerunning final governance reconciliation with --final-calibration-manifest.')
else:
    print('HELD: Runner appears available, but calibration was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit non-claim acknowledgement if appropriate.')
print('
NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
